# Gaps and Overlaps alpha testing
The Notebook contains prototype testing for S-57 CWI data coverage ingest and overlap analysis, including clean-up within ArcGIS Pro and ArcPy

# S-57 import script
Import a raw S-57 ENC file (*.000) into an arc maritime geodatabase, then extract the product covrage to a non-maritime geodatabase - DON'T THINK WE NEED THE SEPERATE GDB as can extract product coverage straight from maritime gdb
The S-57 file must be in an exchange set with the DMD ID in the path name, i.e. //203976_US5AK96M_000//US5AK96M.000, so that the DMD ID can be resolved. If GQC is used as a source for product coverage and DMD ID is included, then this will not be needed. However, it may be useful for S-101 ENC ingest as there is not an instance of the GQC for S-101


In [63]:
# clean normal database

normal_gdb = "G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\CWI_GDB_trial.gdb"

# Set the workspace
arcpy.env.workspace = normal_gdb

arcpy.management.DeleteFeatures('ProductCoverage')

<Result 'ProductCoverage'>

In [ ]:
# Need to write code to clean up maritime gdb after each run
# May delete and recreate everytime

import arcpy
 
def delete_tables(workspace):
    arcpy.env.workspace = workspace
    tables = arcpy.ListTables()
    for table in tables:
        arcpy.Delete_management(table)
       
if __name__ == "__main__":
    import sys
    workspace = sys.argv[1]
    delete_tables(workspace)

In [6]:
"""

Working test for loading S-57 M_COVR into a standard gdb

AJW

v.1.0



"""
import arcpy
import glob


# Set the workspace
arcpy.env.workspace = "G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\Maritime_GDB.gdb"

# Define the S-57 ENC file and the target maritime GDB
s57_enc_file = "G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\Test_Input\\213012_US4AK5PM_000\\US4AK5PM.000"
maritime_gdb = "G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\Maritime_GDB.gdb"
maritime_gdb2 = "G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\GDB_trialgdb\\Maritime_GDB_test_2.gdb"
maritime_gdb3 = "G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\GDB_trialgdb\\Maritime_GDB_test_3.gdb"

config_loc = "G:\IC-ENC USERS\Alex Wallage\Gaps and Overlaps Diss\config xml editing\\NAUTICAL_ENC_TEMPLATE_GX_EXTCLSID_attempt1"


s57LoadDir = glob.glob("G://IC-ENC USERS//Alex Wallage//Gaps and Overlaps Diss//TestData//**//*.000", recursive=True)


# Import the S-57 ENC to the maritime GDB
arcpy.maritime.ImportS57ToGeodatabase(s57_enc_file, maritime_gdb3, in_product_config=config_loc)

# Define the output normal GDB
normal_gdb = "G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\CWI_GDB_trial.gdb"

# Extract DataCoverage features to the normal GDB
data_coverage_features = "ProductCoverage"
arcpy.FeatureClassToFeatureClass_conversion(
    f"{maritime_gdb}\\{data_coverage_features}",
    normal_gdb,
    data_coverage_features
)

print("DataCoverage features have been successfully extracted to the normal GDB.")

DataCoverage features have been successfully extracted to the normal GDB.


In [2]:
import geopandas as gpd
import matplotlib.pyplot as plt
import glob


# Define the path to the normal GDB and the DataCoverage feature class
normal_gdb = "G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\CWI_GDB_trial.gdb"
data_coverage_fc = f"{normal_gdb}\\ProductCoverage"
s57_enc_file = "G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\Test_Input\\213012_US4AK5PM_000\\US4AK5PM.000"


s57LoadDir = glob.glob("G://IC-ENC USERS//Alex Wallage//Gaps and Overlaps Diss//TestData//**//*.000", recursive=True)

dmdIds = []
for i in s57LoadDir:
    dmdId = i.split("\\")[1][0:6] # currently aboslute on DMD len, change to adaptbale using re
    dmdIds.append(dmdId)

#print(dmdIds)


# read DMD ID - may need to loop in final script depending on approach
dmdId = s57_enc_file.split("\\")[6][0:6]

# Define the output path for the Shapefile or GeoPackage
output_shapefile = "G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\ProductCoverage.shp"

# Export to Shapefile
arcpy.FeatureClassToShapefile_conversion([data_coverage_fc], "G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial")

# Read the Shapefile
gdf = gpd.read_file(output_shapefile)


## Re-structure GDB

# filter M_COVR for coverage available
gdf = gdf[gdf['CATCOV'] == 1]

#add DMD_ID - may need to loop in final script depending on approach
gdf['DMD_ID'] = dmdIds # dmdId needs dissolve but dissolve needs DMD_ID work this out - look at how it was done in QGIS

# Dissolve based on DMD_ID
# ADD


#rename fields
gdf =gdf.rename(columns={
                        'DSNM':'CELLNAME', 
                        'PLTS_COMP_':'SCALE'}
               )

# ADD NAVBAND
gdf['NAVBAND'] = gdf['CELLNAME'].astype(str).str[2:3].astype(int)

# reorder gdf
gdf = gdf[['DMD_ID','CELLNAME','SCALE', 'NAVBAND', 'geometry']]

# dissolve on DMD ID to deal with multiparts
gdf =gdf.dissolve(by='DMD_ID')




## JUST FOR TESTING
print(gdf.columns)
print(gdf.head)
#print(gdf.iloc[0]['geometry'])

geometry = gdf.iloc[0]['geometry']

# Check if the geometry is a MultiPolygon
if geometry.geom_type == 'MultiPolygon':
    # Count the number of polygons in the MultiPolygon
    num_polygons = len(geometry.geoms)
    print(f"The MultiPolygon geometry contains {num_polygons} polygons.")
else:
    print("The geometry is not a MultiPolygon.")
    
    
#print(gdf.to_string())

#gdf.plot()

# Add title and labels
#plt.title('Simple GeoDataFrame Plot')
#plt.xlabel('Longitude')
#plt.ylabel('Latitude')


#plt.show()


gdf.to_file("G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\ProductCoverage1.shp")

# would now upload gdf to SQL CWI db

Index(['geometry', 'CELLNAME', 'SCALE', 'NAVBAND'], dtype='object')
<bound method NDFrame.head of                                                  geometry  ... NAVBAND
DMD_ID                                                     ...        
240747  POLYGON ((33.08955 34.69667, 33.0897 34.69667,...  ...       4
240748  POLYGON ((1.57056 52.09988, 1.57056 52.11146, ...  ...       3
240770  POLYGON ((-44 -23.0368, -44 -23.02398, -44 -23...  ...       5
240777  POLYGON ((-95.35 15.94167, -95.35 16.01752, -9...  ...       4
240778  POLYGON ((-95.25417 16.09167, -95.25417 16.096...  ...       5
240779  POLYGON ((-95.21944 16.13889, -95.21944 16.148...  ...       6
240785  POLYGON ((153.3044 -27.05311, 153.30432 -27.05...  ...       6
240787  POLYGON ((146.8144 -19.26556, 146.8144 -19.264...  ...       5

[8 rows x 4 columns]>
The geometry is not a MultiPolygon.


# Overlap Analysis
perform overlap assessment on CWI against Data on the market (allReleased), and other cells in work flow (open CWIs). then filter the overlaps where the NAVBAND is the same OR the SCALE is the same.

Join with DMD for Modification-type ahead of this?

In [60]:
"""
Overlap Analysis

Cartesian buffering

AJW

v.1.0


"""


import geopandas as gpd
import pandas as pd

# Define the output path for the Shapefile or GeoPackage
CWI_path = "G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\ProductCoverage1.shp"

allReleased = "V:\\MANAGEMENT\\IC-ENC Graphical Catalogue\\2025\\CATALOGUE WK0125\\ALLRELEASED.shp"

openCWI = "V:\\MANAGEMENT\\IC-ENC Graphical Catalogue\\In work/TEST\\Fixed Geometries\\Fixed IC-ENC WEEKLY CELLS.shp"


## Read Shapefiles
gdf = gpd.read_file(CWI_path)

allReleasedGdf = gpd.read_file(allReleased)

openCWIGdf = gpd.read_file(openCWI)




## create Overlaps

def check_overlaps(CWI, gdb):
    
    # Check for overlaps
    overlaps = gpd.overlay(CWI, gdb, how='intersection')
    
    return overlaps

overlapAllRelGdf = check_overlaps(gdf, allReleasedGdf)
overlapOCWIGdf = check_overlaps(gdf, openCWIGdf)



# filter overlaps - TEST with diff NAVBANDs same SCALE
def filter_policy(gdf):
    overlapGdf = gdf[
        (gdf['NAVBAND'].astype(int) == gdf['NAV_BAND'].astype(int)) | 
        (gdf['SCALE_1'].astype(int) == gdf['SCALE_2'].astype(int))
    ]
    return overlapGdf
    
overlapAllRelGdf = filter_policy(overlapAllRelGdf)

overlapOCWIGdf = filter_policy(overlapOCWIGdf)


# join overlap GDBs

overlapGdf = pd.concat([overlapAllRelGdf, overlapOCWIGdf], ignore_index=True)

overlapGdf['NAMEJOIN'] = overlapGdf['CELLNAME_1'].astype(str) + overlapGdf['CELLNAME_2'].astype(str)


# remove duplicates - potentially by CWI and 
#DECIDE WHETHER THIS IS NECESSARY, IF WE HAVE INSTANCES IN THE OPEN CWI, WE DON'T THE ALLRELEASED RECORD

# Just NAMEJOIN may be too broad
overlapGdf = overlapGdf.drop_duplicates(subset=['NAMEJOIN'])


## Add Fields (STATUS, REASON, ETC.)
overlapGdf['STATUS'] = ""


#ASSIGN UNIQUE OVERLAP ID
overlapGdf = overlapGdf.assign(unique_id=range(1, len(overlapGdf) + 1))

print(overlapGdf.columns)

## perform autoclassification analysis

def autoclassify_overlaps(CWI, overlap):
    
    CWI = CWI[['CELLNAME', 'geometry']]
    overlap = overlap.rename(columns={'CELLNAME_1' : 'CELLNAME', 
                                      'geometry' : 'geometryOverlap'
                                    }
                            )
    
    CWI = CWI.rename(columns={'geometry' : 'geometryCWI'})
    
    overlap = overlap.merge(CWI, on='CELLNAME',how='left')
    overlap = overlap.drop_duplicates(subset=['unique_id'])

    
    overlap = overlap.set_geometry('geometryCWI')
    
    overlap = overlap.to_crs(epsg=3395)
    
    overlap['buffer'] = overlap['geometryCWI'].buffer(5, cap_style=2).difference(overlap['geometryCWI'])
    
    overlap = overlap.to_crs(epsg=4326)
    
    buffer = gpd.GeoDataFrame(overlap[['buffer','CELLNAME','CELLNAME_2','unique_id']], geometry='buffer')
    
    buffer.to_file('G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\bufferCartesian.shp')
    
    overlapTest = gpd.GeoDataFrame(overlap[['geometryOverlap', 'CELLNAME','CELLNAME_2', 'unique_id']], geometry='geometryOverlap')
    overlapTest.to_file('G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\overlaptest.shp')
    
    #end
    classifiedOverlaps = overlap
    
    return classifiedOverlaps


overlapGdf = autoclassify_overlaps(gdf, overlapGdf)




# TESTING check results
overlapGdf.head

overlapGdf.columns

#overlapGdf.to_file("G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\Overlaps.shp")

#would now upload to overlaps SQL db

Index(['DMD_ID', 'CELLNAME_1', 'SCALE_1', 'NAVBAND', 'CELLNAME_2', 'CELLTITLE',
       'NAV_BAND', 'ED_DATE', 'ED_NO', 'UP_DATE', 'UP_NO', 'SCALE_2',
       'PRODUCER', 'RENC_MEMBE', 'USAGEBAND', 'geometry', 'DMD_ID_1', 'id',
       'DMD_ID_2', 'MODIFIC', 'layer', 'path', 'NAMEJOIN', 'STATUS',
       'unique_id'],
      dtype='object')
9
8
8


[60]:117: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.


Index(['DMD_ID', 'CELLNAME', 'SCALE_1', 'NAVBAND', 'CELLNAME_2', 'CELLTITLE',
       'NAV_BAND', 'ED_DATE', 'ED_NO', 'UP_DATE', 'UP_NO', 'SCALE_2',
       'PRODUCER', 'RENC_MEMBE', 'USAGEBAND', 'geometryOverlap', 'DMD_ID_1',
       'id', 'DMD_ID_2', 'MODIFIC', 'layer', 'path', 'NAMEJOIN', 'STATUS',
       'unique_id', 'geometryCWI', 'buffer'],
      dtype='object')

In [43]:
"""
Overlap Analysis

Ellipsoidal buffering

AJW

v.1.0


"""

import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon
from shapely.ops import transform
import pyproj

# Define the paths to the input shapefiles
CWI_path = "G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\ProductCoverage1.shp"
allReleased = "V:\\MANAGEMENT\\IC-ENC Graphical Catalogue\\2025\\CATALOGUE WK0125\\ALLRELEASED.shp"
openCWI = "V:\\MANAGEMENT\\IC-ENC Graphical Catalogue\\In work/TEST\\Fixed Geometries\\Fixed IC-ENC WEEKLY CELLS.shp"

# Read the shapefiles into GeoDataFrames
gdf = gpd.read_file(CWI_path)
allReleasedGdf = gpd.read_file(allReleased)
openCWIGdf = gpd.read_file(openCWI)

# Function to check for overlaps
def check_overlaps(CWI, gdb):
    overlaps = gpd.overlay(CWI, gdb, how='intersection')
    return overlaps

# Check for overlaps
overlapAllRelGdf = check_overlaps(gdf, allReleasedGdf)
overlapOCWIGdf = check_overlaps(gdf, openCWIGdf)

# Function to filter overlaps based on NAVBAND and SCALE
def filter_policy(gdf):
    overlapGdf = gdf[
        (gdf['NAVBAND'].astype(int) == gdf['NAV_BAND'].astype(int)) | 
        (gdf['SCALE_1'].astype(int) == gdf['SCALE_2'].astype(int))
    ]
    return overlapGdf

# Filter overlaps
overlapAllRelGdf = filter_policy(overlapAllRelGdf)
overlapOCWIGdf = filter_policy(overlapOCWIGdf)

# Join the overlap GeoDataFrames
overlapGdf = pd.concat([overlapAllRelGdf, overlapOCWIGdf], ignore_index=True)
overlapGdf['NAMEJOIN'] = overlapGdf['CELLNAME_1'].astype(str) + overlapGdf['CELLNAME_2'].astype(str)

# Remove duplicates
overlapGdf = overlapGdf.drop_duplicates(subset=['NAMEJOIN'])

# Add STATUS field
overlapGdf['STATUS'] = ""

overlapGdf = overlapGdf.assign(unique_id=range(1, len(overlapGdf) + 1))

# Function to create a geodesic buffer in a Cartesian projection
def geodesic_buffer(geometry, distance):
    centroid = geometry.centroid
    x, y = centroid.x, centroid.y
    
    # Azimuthal equidistant projection string
    proj_string = f"+proj=aeqd +ellps=WGS84 +lat_0={y} +lon_0={x} +x_0=0 +y_0=0"
    
    # Define the source and destination CRS
    src_crs = pyproj.CRS("EPSG:4326")
    dest_crs = pyproj.CRS(proj_string)
    
    # Create transformers for forward and reverse transformations
    transformer_to_aeqd = pyproj.Transformer.from_crs(src_crs, dest_crs, always_xy=True)
    transformer_to_wgs84 = pyproj.Transformer.from_crs(dest_crs, src_crs, always_xy=True)
    
    # Project to azimuthal equidistant projection
    projected_geom = transform(transformer_to_aeqd.transform, geometry)
    
    # Create buffer in projected coordinates
    buffer_geom = projected_geom.buffer(distance, cap_style=2)  # cap_style=2 for flat buffer
    
    # Transform buffer back to WGS84
    buffer_geom_wgs84 = transform(transformer_to_wgs84.transform, buffer_geom)
    
    return buffer_geom_wgs84

# Function to autoclassify overlaps and create donut geometry
def autoclassify_overlaps(CWI, overlap):
    CWI = CWI[['CELLNAME', 'geometry']]
    overlap = overlap.rename(columns={'CELLNAME_1': 'CELLNAME', 'geometry': 'geometryOverlap'})
    CWI = CWI.rename(columns={'geometry': 'geometryCWI'})
    
    overlap = overlap.merge(CWI, on='CELLNAME',how='left')
    overlap = overlap.drop_duplicates(subset=['unique_id'])
    
    # Set the geometry to geometryOverlap for CRS check and buffering
    overlap = overlap.set_geometry('geometryOverlap')
    
    # Ensure the CRS is in a geographic coordinate system for geodesic buffering
    if not overlap.crs.is_geographic:
        overlap['geometryOverlap'] = overlap['geometryOverlap'].to_crs(epsg=4326)
        overlap['geometryCWI'] = overlap['geometryCWI'].to_crs(epsg=4326)
    
    # Create buffers and compute the differences
    distances = [-5, -1, -0.1]
    for distance in distances:
        buffer_name = f'buffer_{abs(distance)}m'
        overlap[buffer_name] = overlap.apply(lambda row: geodesic_buffer(row['geometryCWI'], 0.1).difference(geodesic_buffer(row['geometryCWI'], distance)), axis=1)
    
    # Create the output GeoDataFrame with the correct CRS and tidy up the output
    output_columns = ['CELLNAME'] + [f'buffer_{abs(distance)}m' for distance in distances]
    
    # Create a GeoDataFrame for each buffer and save them separately
    for distance in distances:
        buffer_name = f'buffer_{abs(distance)}m'
        buffer_gdf = gpd.GeoDataFrame(overlap[['CELLNAME', buffer_name]], geometry=buffer_name, crs="EPSG:4326")
        #buffer_gdf.to_file(f'G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\buffer_{abs(distance)}m.shp')
    
    classifiedOverlaps = overlap
    
    return classifiedOverlaps

# Autoclassify overlaps and create donut geometry
overlapGdf = autoclassify_overlaps(gdf, overlapGdf)
print(overlapGdf.columns)

print("Buffer geometries created and saved to shapefiles.")




Index(['DMD_ID', 'CELLNAME', 'SCALE_1', 'NAVBAND', 'CELLNAME_2', 'CELLTITLE',
       'NAV_BAND', 'ED_DATE', 'ED_NO', 'UP_DATE', 'UP_NO', 'SCALE_2',
       'PRODUCER', 'RENC_MEMBE', 'USAGEBAND', 'geometryOverlap', 'DMD_ID_1',
       'id', 'DMD_ID_2', 'MODIFIC', 'layer', 'path', 'NAMEJOIN', 'STATUS',
       'geometryCWI', 'buffer_5m', 'buffer_1m', 'buffer_0.1m'],
      dtype='object')
Buffer geometries created and saved to shapefiles.


In [62]:
"""
Overlap Analysis

Ellipsoidal buffering with autoclassifications

AJW

v.1.0


"""

import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon
from shapely.ops import transform
import pyproj

# Define the paths to the input shapefiles
CWI_path = "G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\ProductCoverage1.shp"
allReleased = "V:\\MANAGEMENT\\IC-ENC Graphical Catalogue\\2025\\CATALOGUE WK0125\\ALLRELEASED.shp"
openCWI = "V:\\MANAGEMENT\\IC-ENC Graphical Catalogue\\In work/TEST\\Fixed Geometries\\Fixed IC-ENC WEEKLY CELLS.shp"

# Read the shapefiles into GeoDataFrames
gdf = gpd.read_file(CWI_path)
allReleasedGdf = gpd.read_file(allReleased)
openCWIGdf = gpd.read_file(openCWI)


# Function to check for overlaps
def check_overlaps(CWI, gdb):
    overlaps = gpd.overlay(CWI, gdb, how='intersection')
    return overlaps

# Check for overlaps
overlapAllRelGdf = check_overlaps(gdf, allReleasedGdf)
overlapOCWIGdf = check_overlaps(gdf, openCWIGdf)

# Function to filter overlaps based on NAVBAND and SCALE
def filter_policy(gdf):
    overlapGdf = gdf[
        (gdf['NAVBAND'].astype(int) == gdf['NAV_BAND'].astype(int)) | 
        (gdf['SCALE_1'].astype(int) == gdf['SCALE_2'].astype(int))
    ]
    return overlapGdf

# Filter overlaps
overlapAllRelGdf = filter_policy(overlapAllRelGdf)
overlapOCWIGdf = filter_policy(overlapOCWIGdf)

# Join the overlap GeoDataFrames
overlapGdf = pd.concat([overlapAllRelGdf, overlapOCWIGdf], ignore_index=True)
overlapGdf['NAMEJOIN'] = overlapGdf['CELLNAME_1'].astype(str) + overlapGdf['CELLNAME_2'].astype(str)

# Remove duplicates
overlapGdf = overlapGdf.drop_duplicates(subset=['NAMEJOIN'])

# Add STATUS field
overlapGdf['STATUS'] = ""

overlapGdf = overlapGdf.assign(unique_id=range(1, len(overlapGdf) + 1))


# Function to create a geodesic buffer in a Cartesian projection
def geodesic_buffer(geometry, distance):
    centroid = geometry.centroid
    x, y = centroid.x, centroid.y
    
    # Azimuthal equidistant projection string
    proj_string = f"+proj=aeqd +ellps=WGS84 +lat_0={y} +lon_0={x} +x_0=0 +y_0=0"
    
    # Define the source and destination CRS
    src_crs = pyproj.CRS("EPSG:4326")
    dest_crs = pyproj.CRS(proj_string)
    
    # Create transformers for forward and reverse transformations
    transformer_to_aeqd = pyproj.Transformer.from_crs(src_crs, dest_crs, always_xy=True)
    transformer_to_wgs84 = pyproj.Transformer.from_crs(dest_crs, src_crs, always_xy=True)
    
    # Project to azimuthal equidistant projection
    projected_geom = transform(transformer_to_aeqd.transform, geometry)
    
    # Create buffer in projected coordinates
    buffer_geom = projected_geom.buffer(distance, cap_style=2)  # cap_style=2 for flat buffer
    
    # Transform buffer back to WGS84
    buffer_geom_wgs84 = transform(transformer_to_wgs84.transform, buffer_geom)
    
    return buffer_geom_wgs84



# Function to autoclassify overlaps and create donut geometry
def autoclassify_overlaps(CWI, overlap):
    CWI = CWI[['CELLNAME', 'geometry']]
    overlap = overlap.rename(columns={'CELLNAME_1': 'CELLNAME', 'geometry': 'geometryOverlap'})
    CWI = CWI.rename(columns={'geometry': 'geometryCWI'})
    
    overlap = overlap.merge(CWI, on='CELLNAME',how='left')
    overlap = overlap.drop_duplicates(subset=['unique_id'])
    
    # Set the geometry to geometryOverlap for CRS check and buffering
    overlap = overlap.set_geometry('geometryOverlap')
    
    # Ensure the CRS is in a geographic coordinate system for geodesic buffering
    if not overlap.crs.is_geographic:
        overlap['geometryOverlap'] = overlap['geometryOverlap'].to_crs(epsg=4326)
        overlap['geometryCWI'] = overlap['geometryCWI'].to_crs(epsg=4326)
    
    # Create buffers and compute the differences
    distances = [-5, -1, -0.1]
    for distance in distances:
        buffer_name = f'buffer_{abs(distance)}m'
        overlap[buffer_name] = overlap.apply(lambda row: geodesic_buffer(row['geometryCWI'], 0.1).difference(geodesic_buffer(row['geometryCWI'], distance)), axis=1)
    
    # Check if geometryOverlap is within each of the buffers and update STATUS field accordingly
    def update_status(row):
        if row['geometryOverlap'].within(row['buffer_0.1m']):
            return 'RESIDUAL'
        elif row['geometryOverlap'].within(row['buffer_1m']):
            return '1mOverlap'
        elif row['geometryOverlap'].within(row['buffer_5m']):
            return '5mOverlap'
        else:
            return ''
    
    overlap['STATUS'] = overlap.apply(update_status, axis=1)
    
    # Create the output GeoDataFrame with the correct CRS and tidy up the output
    output_columns = ['CELLNAME', 'STATUS'] + [f'buffer_{abs(distance)}m' for distance in distances]
    
    # Create a GeoDataFrame for each buffer and save them separately - JUST FOR TESTING
    for distance in distances:
        buffer_name = f'buffer_{abs(distance)}m'
        buffer_gdf = gpd.GeoDataFrame(overlap[['CELLNAME', 'STATUS', buffer_name]], geometry=buffer_name, crs="EPSG:4326")
        buffer_gdf.to_file(f'G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\buffer_{abs(distance)}m.shp')
    
    overlapTest = gpd.GeoDataFrame(overlap[['geometryOverlap', 'CELLNAME','CELLNAME_2', 'unique_id', 'STATUS']], geometry='geometryOverlap')
    overlapTest.to_file('G:\\IC-ENC USERS\\Alex Wallage\\Gaps and Overlaps Diss\\GDB_trial\\autoclassificationtest.shp')
    
    #end
    classifiedOverlaps = overlap
    
    return classifiedOverlaps

# Autoclassify overlaps and create donut geometry
overlapGdf = autoclassify_overlaps(gdf, overlapGdf)





[62]:135: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
[62]:138: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.


Donut shaped geometries created and saved to shapefiles.
